In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    print("Debug: Prompt for dataset generation: ", prompt)
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    print(text)
    return json.loads(text)
    

In [13]:
dataset = generate_dataset()

type(dataset)
print(dataset)

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


Debug: Prompt for dataset generation:  
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.


[
    {
        "task": "Write a Python function that extracts the AWS account ID from an ARN string like 'arn:aws:s3:::my-bucket/key'"
    },
    {
        "task": "Create a JSON object that represents an IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'"
    },
    {
        "task": "Write a regular expression that matches valid A

In [3]:
def run_prompt(test_case):
    """Merges the prompt and test case, and runs the prompt through the model."""
    prompt = f"""
Please solve the following task.
Task: {test_case['task']}"""

    messages = []
    
    add_user_message(messages, prompt)
    output = chat(messages)
    return output.strip()

In [4]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
You are an expert code reviewer. You need to evaluate this AI-generated solution:

Task: {test_case['task']}
Solution: {output}

Provide your evaluation as structured JSON with the following fields:
- "strength": An array of 1-3 key strengths of the solution.
- "weakness": An array of 1-3 key weaknesses of the solution.
- "reasoning": A brief explanation of your evaluation.
- "score": A score from 1 to 10, where 1 is poor and 10 is excellent.
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_output = chat(messages, stop_sequences=["```"])
    return json.loads(eval_output)

In [5]:
def run_test_case(test_case):
    """Runs a single test case and returns the result."""
    # print(f"Running test case: {test_case['task']}")
    output = run_prompt(test_case)

    # Grade the output using the model
    model_grade = grade_by_model(test_case, output)
    score = model_grade['score']
    reasoning = model_grade['reasoning']
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [6]:
from statistics import mean

def run_eval(dataset):
    """Runs the evaluation on the dataset and returns the results."""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    average_score = mean(result['score'] for result in results)
    print(f"Average score across all test cases: {average_score}")    
    
    return results

In [7]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

NameError: name 'json' is not defined

In [19]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Account ID Extraction from ARN\n\nHere's a Python function that extracts the AWS account ID from an ARN string:\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str | None:\n    \"\"\"\n    Extract the AWS account ID from an ARN string.\n    \n    ARN format: arn:partition:service:region:account-id:resource\n    \n    Args:\n        arn: The ARN string to parse\n        \n    Returns:\n        The account ID (12-digit number) or None if not found/invalid\n        \n    Examples:\n        >>> extract_account_id_from_arn('arn:aws:iam::123456789012:user/username')\n        '123456789012'\n        \n        >>> extract_account_id_from_arn('arn:aws:s3:::my-bucket/key')\n        None  # S3 bucket ARNs don't have account IDs\n    \"\"\"\n    try:\n        # Split the ARN by colons\n        parts = arn.split(':')\n        \n        # ARN format: arn:partition:service:region:account-id:resource\n        # Index 4 contains the account ID\n        if len(parts